In [3]:
import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\research


In [4]:
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01")

In [5]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01\\research'

In [6]:
os.chdir("../")

In [7]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01'

In [8]:
import box
print(box.__version__)

7.4.1


In [9]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: Path
    unzip_data_dir: Path
    all_schema: dict

In [10]:
from Regression_01.entity.config_entity import DataValidationConfig
from Regression_01.utils.common import create_directories

In [11]:
# cofiguration manager
from Regression_01.config.configuration import ConfigurationManager

def get_data_validation_config(self) -> DataValidationConfig:

    config = self.config.data_validation

    create_directories([config.root_dir])

    data_validation_config = DataValidationConfig(

        root_dir=config.root_dir,

        STATUS_FILE=config.STATUS_FILE,

        unzip_data_dir=config.unzip_data_dir,

        all_schema=config.all_schema

    )

    return data_validation_config

In [12]:
import pandas as pd
from Regression_01.logging import logger

[2026-07-30 01:06:27,376: INFO: utils: NumExpr defaulting to 4 threads.]


In [13]:
from Regression_01.entity.config_entity import DataValidationConfig

In [14]:
# Components 

class DataValidation:

    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_dataset(self):

        validation_status = True

        # Read Dataset
        if not os.path.exists(self.config.unzip_data_dir):
            logger.info("Dataset Not Found")
            return False

        df = pd.read_csv(self.config.unzip_data_dir)

        # Dataset Shape
        rows, columns = df.shape
        logger.info(f"Rows : {rows}")
        logger.info(f"Columns : {columns}")

        if rows == 0:
            logger.info("Dataset is Empty")
            validation_status = False

        # Expected Columns and Data Types
        expected_dtype = {
            "price": "int64",
            "area": "int64",
            "bedrooms": "int64",
            "bathrooms": "int64",
            "stories": "int64",
            "mainroad": "object",
            "guestroom": "object",
            "basement": "object",
            "hotwaterheating": "object",
            "airconditioning": "object",
            "parking": "int64",
            "prefarea": "object",
            "furnishingstatus": "object"
        }

        # Schema Validation
        expected_columns = list(expected_dtype.keys())
        actual_columns = list(df.columns)

        if expected_columns == actual_columns:
            logger.info("Schema Validation Passed")
        else:
            logger.info("Schema Validation Failed")
            validation_status = False

        # Missing Values
        missing_values = df.isnull().sum()
        logger.info(missing_values)

        if missing_values.sum() > 0:
            logger.info("Missing Values Found")
            validation_status = False

        # Duplicate Rows
        duplicates = df.duplicated().sum()
        logger.info(f"Duplicate Rows : {duplicates}")

        if duplicates > 0:
            validation_status = False

        # Data Type Validation
        logger.info(df.dtypes)

        actual_dtype = df.dtypes.astype(str).to_dict()

        for column, dtype in expected_dtype.items():
            if actual_dtype[column] != dtype:
                logger.info(f"{column} datatype mismatch")
                validation_status = False

        # Target Column Check
        if "price" not in df.columns:
            logger.info("Target Column Missing")
            validation_status = False

        # Yes / No Column Validation
        yes_no_columns = [
            "mainroad",
            "guestroom",
            "basement",
            "hotwaterheating",
            "airconditioning",
            "prefarea"
        ]

        for col in yes_no_columns:
            if not df[col].isin(["yes", "no"]).all():
                logger.info(f"{col} contains invalid values")
                validation_status = False

        # Furnishing Status Validation
        valid_furnishing = [
            "furnished",
            "semi-furnished",
            "unfurnished"
        ]

        if not df["furnishingstatus"].isin(valid_furnishing).all():
            logger.info("Invalid furnishingstatus values")
            validation_status = False

        # Negative Value Check
        numeric_columns = [
            "price",
            "area",
            "bedrooms",
            "bathrooms",
            "stories",
            "parking"
        ]

        for col in numeric_columns:
            if (df[col] < 0).any():
                logger.info(f"{col} contains negative values")
                validation_status = False

        # Write Validation Status
        with open(self.config.STATUS_FILE, "w") as f:
            f.write(f"Validation Status : {validation_status}")

        logger.info(f"Validation Status : {validation_status}")

        return validation_status
        
            

In [15]:
# pipeline 

STAGE_NAME="DATA VALIDATION STAGE"

class DataValidationTrainingPipeline:

    def main(self):

        config = ConfigurationManager()

        validation_config = config.get_data_validation_config()

        validation = DataValidation(validation_config)

        validation.validate_dataset()

In [16]:
obj = DataValidationTrainingPipeline()

obj.main()

[2026-07-30 01:06:35,035: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\config\config.yaml loaded successfully]
[2026-07-30 01:06:35,043: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\params.yaml loaded successfully]
[2026-07-30 01:06:35,050: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\schema.yaml loaded successfully]
[2026-07-30 01:06:35,053: INFO: common: created directory at artifacts]
[2026-07-30 01:06:35,057: INFO: common: created directory at artifacts/data_validation]
[2026-07-30 01:06:35,081: INFO: 3930163393: Rows : 545]
[2026-07-30 01:06:35,083: INFO: 3930163393: Columns : 13]
[2026-07-30 01:06:35,085: INFO: 3930163393: Schema Validation Passed]
[2026-07-30 01:06:35,090: INFO: 3930163393: price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement           